In [11]:
import os
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, f1_score, precision_score, recall_score
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import torch.nn as nn
import torch.optim as optim

In [12]:
root_dir = "data"
csv_path = os.path.join(root_dir, "metadata.csv")
num_epochs = 10
batch_size = 16
lr = 1e-4
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [13]:
class ImageDataset(Dataset):
    def __init__(self, df, root_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.root_dir = root_dir
        self.transform = transform
        self.label2idx = {label: idx for idx, label in enumerate(sorted(df["label"].unique()))}
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        img_path = os.path.join(self.root_dir, self.df.loc[idx, "image_path"])
        label = self.label2idx[self.df.loc[idx, "label"]]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label

In [18]:
df = pd.read_csv(csv_path)
# 按照 8:1:1 划分：训练集80%，验证集10%，测试集10%
# 首先分出训练集和临时集（80%训练，20%验证+测试）
train_df, val_test_df = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=42)
# 再从验证+测试集中分出验证集和测试集（各50%，相对于原始数据是各10%）
val_df, test_df = train_test_split(val_test_df, test_size=0.5, stratify=val_test_df["label"], random_state=42)

print(f"训练集大小: {len(train_df)} ({len(train_df)/len(df)*100:.1f}%)")
print(f"验证集大小: {len(val_df)} ({len(val_df)/len(df)*100:.1f}%)")
print(f"测试集大小: {len(test_df)} ({len(test_df)/len(df)*100:.1f}%)")

训练集大小: 320 (80.0%)
验证集大小: 40 (10.0%)
测试集大小: 40 (10.0%)


In [19]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [20]:
train_dataset = ImageDataset(train_df, root_dir, transform)
val_dataset = ImageDataset(val_df, root_dir, transform)
test_dataset = ImageDataset(test_df, root_dir, transform)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

# 保存标签映射，用于后续评估
label2idx = train_dataset.label2idx
idx2label = {v: k for k, v in label2idx.items()}

In [21]:
model = models.resnet18(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, len(df["label"].unique()))
model = model.to(device)

/inspire/hdd/project/aiscientist/yedongxin-CZXS25120006/miniconda3/envs/jianxiaozhi_web/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/inspire/hdd/project/aiscientist/yedongxin-CZXS25120006/miniconda3/envs/jianxiaozhi_web/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [22]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=lr)

In [23]:
def train():
    for epoch in range(num_epochs):
        model.train()
        total_loss, correct, total = 0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * labels.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
        
        acc = correct / total * 100
        print(f"[Epoch {epoch+1}] Train Loss: {total_loss/total:.4f}, Train Acc: {acc:.2f}%")

        # 验证
        model.eval()
        val_correct, val_total = 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, preds = torch.max(outputs, 1)
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)
        val_acc = val_correct / val_total * 100
        print(f"           Val Acc: {val_acc:.2f}%\n")
    torch.save(model.state_dict(), os.path.join(root_dir, "router_resnet18.pth"))

In [24]:
train()

# 训练完成后，在测试集上评估
print("\n" + "="*50)
print("开始在测试集上评估模型...")
print("="*50)

[Epoch 1] Train Loss: 0.2123, Train Acc: 92.50%
           Val Acc: 100.00%

[Epoch 2] Train Loss: 0.0065, Train Acc: 100.00%
           Val Acc: 100.00%

[Epoch 3] Train Loss: 0.0050, Train Acc: 100.00%
           Val Acc: 100.00%

[Epoch 4] Train Loss: 0.0059, Train Acc: 100.00%
           Val Acc: 100.00%

[Epoch 5] Train Loss: 0.0046, Train Acc: 100.00%
           Val Acc: 100.00%

[Epoch 6] Train Loss: 0.0063, Train Acc: 100.00%
           Val Acc: 100.00%

[Epoch 7] Train Loss: 0.0021, Train Acc: 100.00%
           Val Acc: 100.00%

[Epoch 8] Train Loss: 0.0027, Train Acc: 100.00%
           Val Acc: 100.00%

[Epoch 9] Train Loss: 0.0015, Train Acc: 100.00%
           Val Acc: 100.00%

[Epoch 10] Train Loss: 0.0069, Train Acc: 99.69%
           Val Acc: 100.00%


开始在测试集上评估模型...


In [27]:
def evaluate_test_set(model, test_loader, device, idx2label, results_dir):
    """
    在测试集上评估模型并保存结果
    
    Args:
        model: 训练好的模型
        test_loader: 测试集数据加载器
        device: 设备
        idx2label: 索引到标签的映射
        results_dir: 结果保存目录
    """
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            probs = torch.nn.functional.softmax(outputs, dim=1)
            _, preds = torch.max(outputs, 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)
    
    # 计算指标
    accuracy = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='weighted')
    precision = precision_score(all_labels, all_preds, average='weighted')
    recall = recall_score(all_labels, all_preds, average='weighted')
    
    # 计算每个类别的指标
    class_f1 = f1_score(all_labels, all_preds, average=None)
    class_precision = precision_score(all_labels, all_preds, average=None)
    class_recall = recall_score(all_labels, all_preds, average=None)
    
    # 混淆矩阵
    cm = confusion_matrix(all_labels, all_preds)
    
    # 创建结果目录
    os.makedirs(results_dir, exist_ok=True)
    
    # 保存文本报告
    report = classification_report(all_labels, all_preds, 
                                   target_names=[idx2label[i] for i in range(len(idx2label))])
    
    metrics_text = f"""
测试集评估结果
{'='*50}

总体指标:
- 准确率 (Accuracy): {accuracy:.4f}
- 加权F1分数 (Weighted F1): {f1:.4f}
- 加权精确率 (Weighted Precision): {precision:.4f}
- 加权召回率 (Weighted Recall): {recall:.4f}

分类报告:
{report}

混淆矩阵:
{cm}
"""
    
    # 保存指标文本文件
    metrics_path = os.path.join(results_dir, 'test_metrics.txt')
    with open(metrics_path, 'w', encoding='utf-8') as f:
        f.write(metrics_text)
    print(f"评估指标已保存到: {metrics_path}")
    
    # 保存JSON格式的指标
    metrics_dict = {
        'accuracy': float(accuracy),
        'weighted_f1': float(f1),
        'weighted_precision': float(precision),
        'weighted_recall': float(recall),
        'per_class_metrics': {
            idx2label[i]: {
                'f1': float(class_f1[i]),
                'precision': float(class_precision[i]),
                'recall': float(class_recall[i])
            }
            for i in range(len(idx2label))
        },
        'confusion_matrix': cm.tolist(),
        'class_names': [idx2label[i] for i in range(len(idx2label))]
    }
    
    metrics_json_path = os.path.join(results_dir, 'test_metrics.json')
    with open(metrics_json_path, 'w', encoding='utf-8') as f:
        json.dump(metrics_dict, f, indent=2, ensure_ascii=False)
    print(f"评估指标JSON已保存到: {metrics_json_path}")
    
    # 保存混淆矩阵CSV
    cm_df = pd.DataFrame(cm, 
                         index=[idx2label[i] for i in range(len(idx2label))],
                         columns=[idx2label[i] for i in range(len(idx2label))])
    cm_csv_path = os.path.join(results_dir, 'confusion_matrix.csv')
    cm_df.to_csv(cm_csv_path)
    print(f"混淆矩阵CSV已保存到: {cm_csv_path}")
    
    # 绘制并保存混淆矩阵图
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues', cbar=True)
    plt.title('Confusion Matrix - Test Set')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    cm_plot_path = os.path.join(results_dir, 'confusion_matrix.png')
    plt.savefig(cm_plot_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"混淆矩阵图已保存到: {cm_plot_path}")
    
    # 打印结果
    print("\n" + "="*50)
    print("测试集评估结果:")
    print(f"准确率: {accuracy:.4f}")
    print(f"加权F1分数: {f1:.4f}")
    print(f"加权精确率: {precision:.4f}")
    print(f"加权召回率: {recall:.4f}")
    print("="*50)
    
    return metrics_dict

# 执行测试集评估
results_dir = "../results/model_router"
test_metrics = evaluate_test_set(model, test_loader, device, idx2label, results_dir)


评估指标已保存到: ../results/model_router/test_metrics.txt
评估指标JSON已保存到: ../results/model_router/test_metrics.json
混淆矩阵CSV已保存到: ../results/model_router/confusion_matrix.csv
混淆矩阵图已保存到: ../results/model_router/confusion_matrix.png

测试集评估结果:
准确率: 1.0000
加权F1分数: 1.0000
加权精确率: 1.0000
加权召回率: 1.0000


### 推理

In [12]:
from PIL import Image
import torch
from torchvision import transforms, models
import os

In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [14]:
label2idx = {'breast_cancer': 0, 'chest_cancer': 1, 'eye_disease': 2, 'skin_cancer': 3}
idx2label = {v: k for k, v in label2idx.items()}

In [15]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [16]:
def load_router_model(model_path):
    model = models.resnet18(pretrained=False)
    model.fc = torch.nn.Linear(model.fc.in_features, len(label2idx))
    model.load_state_dict(torch.load(model_path, map_location=device))
    model = model.to(device)
    model.eval()
    return model

In [33]:
import torch.nn.functional as F
def predict_image(input_data, model, threshold=0.6):
    if isinstance(input_data, str):
        image = Image.open(input_data).convert("RGB")
    elif isinstance(input_data, Image.Image):
        image = input_data.convert("RGB")
    else:
        raise ValueError("Input must be a file path or PIL.Image")

    image_tensor = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(image_tensor)
        probs = F.softmax(outputs, dim=1).cpu().numpy().flatten()
        max_prob = probs.max()
        pred_idx = probs.argmax()
        print(max_prob)
        if max_prob < threshold:
            return "Uncertain / Unknown"
        return idx2label[pred_idx]

In [31]:
model_path = "/home/dxye/Program/PhysicalExaminationAgent/client/risk_assessment/model_router/data/router_resnet18.pth"
model = load_router_model(model_path)

/home/dxye/miniconda3/envs/baichuanm1/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/dxye/miniconda3/envs/baichuanm1/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
/tmp/ipykernel_542692/914182344.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_o

In [38]:
result1 = predict_image("/home/dxye/Program/PhysicalExaminationAgent/client/risk_assessment/model_router/data/eye_disease/eye_disease_001.png", model,threshold=0.90)
print("预测结果为：", result1)

0.9999728
预测结果为： eye_disease
